# LEGACY
- option to expand middle layer + validity layers etc

In [1]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui.widgets import PushButton
from qtpy.QtCore import QTimer

In [2]:
combine_input_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\STUFF_FOR_BEL_ONLY\WIP_Bel_2_channel_images_with_separate_masks")
combine_output_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate")
combine_output_folder.mkdir(parents=True, exist_ok=True)


def pixel_sizes_um(path):
    """Read (z_um, y_um, x_um) pixel sizes from a tif's metadata. Any value that
    is unavailable is returned as None."""
    z_um = y_um = x_um = None
    with tifffile.TiffFile(path) as tif:
        tags = {t.name: t.value for t in tif.pages[0].tags.values()}
        xr = tags.get("XResolution")
        yr = tags.get("YResolution")
        if xr and xr[0]:
            x_um = xr[1] / xr[0]  # XResolution = pixels per unit -> um per pixel
        if yr and yr[0]:
            y_um = yr[1] / yr[0]
        ij = tif.imagej_metadata or {}
        if "spacing" in ij:
            z_um = float(ij["spacing"])
    return z_um, y_um, x_um


def peek_raw(path):
    """Read a tif's array plus the raw shape/axes tifffile reports, with no
    reshaping. Used both for conversion and for error diagnostics."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        arr = np.asarray(series.asarray())
        raw_axes = series.axes.upper()
    return arr, arr.shape, raw_axes


def to_zcyx(arr, raw_axes, name):
    """Reshape a raw array + axis labels into (Z, C, Y, X).

    Singleton dimensions are squeezed away first (a 1-channel segmentation is
    often stored as (Z, 1, Y, X) or (1, Z, Y, X)), then the remaining axes are
    mapped to Z, C, Y, X.

    tifffile labels a bare multi-page stack with a generic axis ('I', 'Q', 'S',
    'T', ...) when the file carries no hyperstack metadata. Any such non-singleton
    axis that isn't already C/Y/X is treated as the Z (stacking) axis.
    """
    if len(raw_axes) != arr.ndim:
        # Axis labels are unreliable: infer purely from the squeezed shape.
        arr = np.squeeze(arr)
        raw_axes = {2: "YX", 3: "ZYX", 4: "ZCYX"}.get(arr.ndim)
        if raw_axes is None:
            raise ValueError(f"Cannot infer axes for shape {arr.shape} ({name})")

    # Normalise unknown/generic axis labels (I, Q, S, T, ...) to Z, since the
    # only non-channel stacking axis we expect is depth.
    axes = "".join(a if a in "ZCYX" else "Z" for a in raw_axes)

    # Keep Y/X always; keep other axes only if they are non-singleton.
    keep_axis = [s > 1 or axes[i] in "YX" for i, s in enumerate(arr.shape)]
    arr = arr[tuple(slice(None) if k else 0 for k in keep_axis)]
    axes = "".join(axes[i] for i, k in enumerate(keep_axis) if k)

    # Add any missing leading axes (C then Z) as size-1 so we always have ZCYX.
    for ax in ("C", "Z"):
        if ax not in axes:
            arr = np.expand_dims(arr, 0)
            axes = ax + axes

    order = [axes.index(a) for a in "ZCYX"]
    return np.transpose(arr, order)


def describe(path):
    """Best-effort raw shape/axes string for diagnostics (never raises)."""
    try:
        _, shape, axes = peek_raw(path)
        return f"raw shape={shape} axes={axes!r}"
    except Exception as exc:
        return f"<could not read: {exc}>"


image_paths = sorted(p for p in combine_input_folder.glob("*.tif") if not p.stem.endswith("_segmentation"))
print(f"{len(image_paths)} images found")

# Skip images whose combined output already exists in the output folder.
already_done = {p.name for p in combine_output_folder.glob("*.tif")}

problems = []   # (filename, reason)
skipped = 0
written = 0

for img_path in image_paths:
    out_path = combine_output_folder / img_path.name

    if img_path.name in already_done:
        skipped += 1
        continue

    seg_path = img_path.with_name(f"{img_path.stem}_segmentation.tif")

    try:
        if not seg_path.exists():
            print(f"[MISSING SEG]  {img_path.name} -> no {seg_path.name}")
            problems.append((img_path.name, "missing segmentation"))
            continue

        # Read raw arrays/axes first and print them BEFORE any reshaping, so the
        # diagnostics are always visible even if the ZCYX conversion fails.
        img_arr, img_raw_shape, img_raw_axes = peek_raw(img_path)
        seg_arr, seg_raw_shape, seg_raw_axes = peek_raw(seg_path)

        print(f"\n{img_path.name}")
        print(f"    image: raw shape={img_raw_shape} axes={img_raw_axes!r}")
        print(f"    seg:   raw shape={seg_raw_shape} axes={seg_raw_axes!r}")

        img = to_zcyx(img_arr, img_raw_axes, img_path.name)   # expect (Z, 2, Y, X)
        seg = to_zcyx(seg_arr, seg_raw_axes, seg_path.name)   # expect (Z, 1, Y, X)

        print(f"    image -> ZCYX {img.shape}")
        print(f"    seg   -> ZCYX {seg.shape}")

        if img.shape[0] != seg.shape[0]:
            print(f"    [Z MISMATCH]  image z={img.shape[0]} vs seg z={seg.shape[0]} -- skipped")
            problems.append((img_path.name, f"z mismatch {img.shape[0]} vs {seg.shape[0]}"))
            continue
        if img.shape[2:] != seg.shape[2:]:
            print(f"    [XY MISMATCH] image yx={img.shape[2:]} vs seg yx={seg.shape[2:]} -- skipped")
            problems.append((img_path.name, f"xy mismatch {img.shape[2:]} vs {seg.shape[2:]}"))
            continue

        # Combine the 2-channel image with the 1-channel segmentation into a
        # single (Z, 3, Y, X) stack for curation.
        out = np.concatenate([img, seg.astype(img.dtype)], axis=1)
        dtype = out.dtype

        # Pixel size comes from the ORIGINAL image, not the segmentation.
        z_um, y_um, x_um = pixel_sizes_um(img_path)
        if y_um is None and x_um is not None:
            y_um = x_um
        if x_um is None or y_um is None:
            print(f"    [WARN] no pixel size in {img_path.name}; writing without resolution")
            tifffile.imwrite(out_path, out, imagej=True, metadata={"axes": "ZCYX"})
        else:
            metadata = {"axes": "ZCYX", "unit": "micron"}
            if z_um:
                metadata["spacing"] = z_um
            tifffile.imwrite(out_path, out, imagej=True,
                             resolution=(1.0 / x_um, 1.0 / y_um), metadata=metadata)
        written += 1
        print(f"    saved -> {out_path.name}  shape={out.shape}  dtype={dtype}")

    except Exception as exc:
        # Print raw shape/axes of BOTH files so axis-ordering issues are obvious.
        print(f"[ERROR]        {img_path.name}: {exc}")
        print(f"    image: {describe(img_path)}")
        print(f"    seg:   {describe(seg_path)}")
        problems.append((img_path.name, f"error: {exc}"))
        continue

print(f"\nDone. {written} written, {skipped} already-done (skipped), "
      f"{len(problems)} flagged.")

if problems:
    print("Flagged files:")
    for name, reason in problems:
        print(f"  - {name}: {reason}")


43 images found

Bel_mCh_BF_VASCUMAP_FL32_ARi4_Merged.tif
    image: raw shape=(25, 2, 5607, 5608) axes='ZCYX'
    seg:   raw shape=(25, 5607, 5608) axes='ZYX'
    image -> ZCYX (25, 2, 5607, 5608)
    seg   -> ZCYX (25, 1, 5607, 5608)
    saved -> Bel_mCh_BF_VASCUMAP_FL32_ARi4_Merged.tif  shape=(25, 3, 5607, 5608)  dtype=uint8

Bel_mCh_BF_VASCUMAP_FL32_UTD4_Merged.tif
    image: raw shape=(31, 2, 4692, 5605) axes='ZCYX'
    seg:   raw shape=(31, 4692, 5605) axes='IYX'
    image -> ZCYX (31, 2, 4692, 5605)
    seg   -> ZCYX (31, 1, 4692, 5605)
    saved -> Bel_mCh_BF_VASCUMAP_FL32_UTD4_Merged.tif  shape=(31, 3, 4692, 5605)  dtype=uint8

Bel_mCh_BF_VASCUMAP_FL33_ARi1_Merged.tif
    image: raw shape=(31, 2, 5606, 5605) axes='ZCYX'
    seg:   raw shape=(31, 5606, 5605) axes='ZYX'
    image -> ZCYX (31, 2, 5606, 5605)
    seg   -> ZCYX (31, 1, 5606, 5605)
    saved -> Bel_mCh_BF_VASCUMAP_FL33_ARi1_Merged.tif  shape=(31, 3, 5606, 5605)  dtype=uint8

Bel_mCh_BF_VASCUMAP_FL33_ARi4_Merged.tif


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\tifffile\tifffile.py:3821: UserWarning: <tifffile.TiffWriter 'day5_6mm_dish1_…vice7_Merged.tif'> truncating ImageJ file
  warnings.warn(


    saved -> day5_6mm_dish1_device7_Merged_day5_6mm_dish1_device7_Merged.tif  shape=(46, 3, 7800, 5172)  dtype=uint16

Farid_Thunder_180226_static_Static_Static_8_Merged.tif
    image: raw shape=(60, 2, 7440, 2857) axes='ZCYX'
    seg:   raw shape=(60, 7440, 2857) axes='IYX'
    image -> ZCYX (60, 2, 7440, 2857)
    seg   -> ZCYX (60, 1, 7440, 2857)


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\tifffile\tifffile.py:3821: UserWarning: <tifffile.TiffWriter 'Farid_Thunder_18…ic_8_Merged.tif'> truncating ImageJ file
  warnings.warn(


    saved -> Farid_Thunder_180226_static_Static_Static_8_Merged.tif  shape=(60, 3, 7440, 2857)  dtype=uint16
[MISSING SEG]  VM_3D_Tina_From_Luca_basal_FGF_4.tif -> no VM_3D_Tina_From_Luca_basal_FGF_4_segmentation.tif

VM_3D_Tina_From_Luca_basal_VEGF_1.tif
    image: raw shape=(103, 2, 3759, 2848) axes='ZCYX'
    seg:   raw shape=(103, 3759, 2848) axes='IYX'
    image -> ZCYX (103, 2, 3759, 2848)
    seg   -> ZCYX (103, 1, 3759, 2848)


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\tifffile\tifffile.py:3821: UserWarning: <tifffile.TiffWriter 'VM_3D_Tina_From…basal_VEGF_1.tif'> truncating ImageJ file
  warnings.warn(


    saved -> VM_3D_Tina_From_Luca_basal_VEGF_1.tif  shape=(103, 3, 3759, 2848)  dtype=uint16
[MISSING SEG]  VM_3D_Tina_From_Luca_basal_VEGF_2.tif -> no VM_3D_Tina_From_Luca_basal_VEGF_2_segmentation.tif
[ERROR]        VM_3D_Tina_From_Luca_basal_VEGF_FGF_1.tif: index 1 out of range(1)
    image: raw shape=(82, 2, 3759, 2848) axes='ZCYX'
    seg:   <could not read: index 1 out of range(1)>

Done. 7 written, 33 already-done (skipped), 3 flagged.
Flagged files:
  - VM_3D_Tina_From_Luca_basal_FGF_4.tif: missing segmentation
  - VM_3D_Tina_From_Luca_basal_VEGF_2.tif: missing segmentation
  - VM_3D_Tina_From_Luca_basal_VEGF_FGF_1.tif: error: index 1 out of range(1)
